In [6]:
import numpy as np
import statsmodels.api as sm
from sklearn.decomposition import PCA
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools
import logging
from one.api import ONE
from brainbox.io.one import SessionLoader
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from collections import defaultdict
import pandas as pd
from manifold.decoding.functions.utils import check_config_decoding
import numpy as np
import pickle as pkl
from manifold.decoding.functions import nulldistributions
from communication_subspace.ibl_communication.utils import load_widefield_epoch
from tqdm import tqdm
from iblatlas.atlas import AllenAtlas
from iblatlas.regions import BrainRegions
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import warnings
from manifold.utils import get_trial_masks
from scipy.stats import pointbiserialr
from manifold.widefield_ppi import beryl_mapping,aggregate_by_parent
from matplotlib import pyplot as plt
from glob import glob
import os

In [11]:
%load_ext autoreload
%autoreload 2

In [12]:
df_all = pd.read_parquet('../data/generated/prior_sig/resultsdf.pqt')

In [13]:
from prior_localization.run_scripts.format_outputs_stage_2 import reformat_df

In [15]:
df = reformat_df(df_all)

In [17]:
df_all

,subject,eid,region,N_units,fold,pseudo_id,run_id,score_test,n_trials
0,CSK-im-010,6f94a278-ee23-43bd-868f-889157db8a8d,V,243,-1,-1,1,-0.010373,707
1,CSK-im-010,6f94a278-ee23-43bd-868f-889157db8a8d,V,243,-1,-1,2,0.000984,707
2,CSK-im-010,6f94a278-ee23-43bd-868f-889157db8a8d,V,243,-1,-1,3,-0.012163,707
3,CSK-im-010,6f94a278-ee23-43bd-868f-889157db8a8d,V,243,-1,-1,4,-0.004261,707
4,CSK-im-010,6f94a278-ee23-43bd-868f-889157db8a8d,V,243,-1,-1,5,-0.003944,707
...,...,...,...,...,...,...,...,...,...
186925,CSK-im-011,d34a502f-bd06-471f-8334-df41f785e1d9,V,243,-1,200,6,-0.016577,643
186926,CSK-im-011,d34a502f-bd06-471f-8334-df41f785e1d9,V,243,-1,200,7,-0.017611,643
186927,CSK-im-011,d34a502f-bd06-471f-8334-df41f785e1d9,V,243,-1,200,8,-0.024650,643
186928,CSK-im-011,d34a502f-bd06-471f-8334-df41f785e1d9,V,243,-1,200,9,-0.036938,643
